In [ ]:
# PySpark Setup - Run this cell first
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install pyspark -q
    !pip install findspark -q
    import findspark
    findspark.init()


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .master("local[*]") \
        .appName('cookbook') \
        .getOrCreate()

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
        .master('local[*]') \
        .appName('learn') \
        .getOrCreate()

In [ ]:
sc = spark.sparkContext

In [ ]:
rdd = sc.parallelize([('Mike', 19), ('June', 18), ('Rachel',16), ('Rob',18), ('Scott', 17)])

In [ ]:
rdd.count()

In [ ]:
rdd.getNumPartitions()

In [ ]:
for key, value in sc.getConf().getAll():
    print(f"{key}: {value}")

In [ ]:
rdd

In [ ]:
w = rdd.collect()

In [ ]:
type(w)

In [ ]:
billing_rdd = sc.textFile('datasets/billing_data.csv')

In [ ]:
billing_rdd.take(3)

In [ ]:
myRDD =  sc \
       .textFile(
           'datasets/airport-codes-na.txt'
           , minPartitions=3
           , use_unicode=True
       ).map(lambda element: element.split("\t"))


In [ ]:
myRDD.take(3)

In [ ]:
myRDD =  sc \
       .textFile(
           'datasets/airport-codes-na.txt') \
        .map(lambda element: element.split("\t"))

In [ ]:
myRDD.take(3)

In [ ]:
myRDD.count()

In [ ]:
myRDD.getNumPartitions()

- A key aspect of partitions for your RDD is that the more partitions you have, the higher the parallelism. Potentially, having more partitions will improve your query performance

In [ ]:
departuredelays_rdd = sc.textFile('datasets/departuredelays.csv') \
                        .map(lambda x: x.split(','))

In [ ]:
departuredelays_rdd.count()

In [ ]:
departuredelays_rdd.getNumPartitions()

In [ ]:
departuredelays_rdd.take(3)

In [ ]:
w = myRDD.map(lambda x: (x[0],x[1])) \
        .filter(lambda x: x[1]=="WA")

In [ ]:
w.take(5)

In [ ]:
t = w.flatMap(lambda x: x)

In [ ]:
t.take(4)

In [ ]:
w = myRDD.map(lambda x:x[2]).distinct()

In [ ]:
w.take(5)

In [ ]:
airports = (
    sc
    .textFile('datasets/airport-codes-na.txt')
    .map(lambda x: x.split('\t'))
)

In [ ]:
airports.take(5)

In [ ]:
flights = (
    sc
    .textFile('datasets/departuredelays.csv')
    .map(lambda x: x.split(","))   
)

In [ ]:
flights.take(5)

In [ ]:
(air.zipWithIndex() 
.filter(lambda x: x[-1]!=0) 
.map(lambda x: x[0]) 
.take(5)
)

In [ ]:
def check(x):
    return (x[0].lower(),x[1])

In [ ]:
air.map(lambda x: check(x)).take(5)

In [ ]:
flt.join(air).take(5)

In [ ]:
flights.map(lambda x: (x[0][::-1],x[1])).take(5)

In [ ]:
flights.map(lambda x: (x[0],x[1])).distinct().count()

In [ ]:
(
    flights.map(lambda x:x[2])
    .distinct()
    .sample(False,0.001,123)
    
).take(500)

In [ ]:
# DataFrame example
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('SparkByExamples.com') \
        .master("local[5]").getOrCreate()

df=spark.range(0,20)
print(df.rdd.getNumPartitions())

df.write.mode("overwrite").csv("partition.csv")

In [ ]:
cDf = spark.createDataFrame([(None, None,3), (1, None,None), (34, 2,3)], ("a", "b","c"))
cDf.show()

In [ ]:
from pyspark.sql.functions import *

In [ ]:
cDf.select(coalesce(cDf["a"], cDf["b"],cDf["c"])).show()

# Some useful Functions

- sc.textFile(): Read all kind of data


# RDD Transformations function

- zipWithIndex(): Provides index starting from 0 to all rows in the RDD
- map(): Selecting columns from your RDD
- flatMap(): RDD flattens out all of the elements (that is, a sequence of events)
- filter(): Running a WHERE (filter) clause 
- distinct() : Getting the distinct values
- sample(): samples a fraction of the data, with or without replacement
- join():
- getNumPartitions() : Getting the number of partitions
- reduceByKey() : The reduceByKey(f) transformation reduces the elements of the RDD using f by the key. 
- groupBy():
- groupByKey():
- sortBy():
- sortByKey():
- union()

- Difference Between repartition() and coalesce()
- coalesce() : Returns the first column that is not null.
- lit(): add a new column to DataFrame by assigning a literal or constant value

In [ ]:
data = [("111",50000),("222",60000),("333",40000)]
columns= ["EmpId","Salary"]
df = spark.createDataFrame(data = data, schema = columns)

In [ ]:
df.show()

In [ ]:
from pyspark.sql.functions import *

In [ ]:
w = "fun"

In [ ]:
df.select("EmpId","Salary",lit(w).alias('col1')).show()

In [ ]:
df.printSchema()

In [ ]:
(
    flights
    .zipWithIndex()
    .filter(lambda x : x[1]>0)
    .map(lambda x: (x[0][3],int(x[0][1])))
    .reduceByKey(lambda x, y: x + y)
    .take(5)
)

In [ ]:
w = (
    flights
    .zipWithIndex()
    .filter(lambda x : x[1]>0)
    .map(lambda x: (x[0][3],int(x[0][1])))
    .filter(lambda x: x[0]=='ACT')
    .map(lambda x:x[1])
    .collect()
)

In [ ]:
w = (
    flights
    .zipWithIndex()
    .filter(lambda x : x[1]>0)
    .map(lambda x: (x[0][3],int(x[0][1])))
    .groupBy(lambda x: x[0])
    .collect()
)

In [ ]:
w = (
    flights
    .zipWithIndex()
    .filter(lambda x : x[1]>0)
    .map(lambda x: (x[0][3],int(x[0][1])))
    .groupByKey()
    .collect()
)

In [ ]:
ww = (
    flights
    .zipWithIndex()
    .filter(lambda x : x[1]>0)
    .map(lambda x: (x[0][3],int(x[0][1])))
    .groupByKey()
    .mapValues(len)
    .collect()
)

In [ ]:
tt = (
    flights
    .zipWithIndex()
    .filter(lambda x : x[1]>0)
    .map(lambda x: (x[0][3],int(x[0][1])))
    .groupByKey()
    .map(lambda x: (x[0],list(x[1])))
    .collect()
)

### Values Stored above in 'ww' and 'tt' Varibales are same, only implementation is different

In [ ]:
(
    flights
    .zipWithIndex()
    .filter(lambda x : x[1]>0)
    .map(lambda x: (x[0][3],int(x[0][1])))
    .groupByKey()
    .mapValues(len)
    .sortBy(lambda x:x[0][-1],False)
    .take(5)
    
)

In [ ]:
(
    flights
    .zipWithIndex()
    .filter(lambda x : x[1]>0)
    .map(lambda x: (x[0][3],int(x[0][1])))
    .groupByKey()
    .mapValues(len)
    .sortByKey()
    .take(5)
    
)

In [ ]:
wa = (
    airports
    .zipWithIndex()
    .filter(lambda x: x[1]>0)
    .map(lambda x: x[0])
    .filter(lambda x: x[1]=="WA")
)

bc = (
    airports
    .zipWithIndex()
    .filter(lambda x: x[1]>0)
    .map(lambda x: x[0])
    .filter(lambda x: x[1]=="BC")
)

In [ ]:
wa.union(bc).collect()

In [ ]:
w = flights.repartition(8)

In [ ]:
flights.count()

In [ ]:
w.getNumPartitions()

In [ ]:
# Source: https://stackoverflow.com/a/38957067/1100699
def partitionElementCount(idx, iterator):
     count = 0
     for _ in iterator:
       count += 1
     return idx, count
# Use mapPartitionsWithIndex to determine
w.mapPartitionsWithIndex(partitionElementCount).collect()


In [ ]:
from pyspark.pandas import read_excel

w = read_excel('datasets/Book.xlsx',sheet_name='Sheet1')

In [ ]:
data = []
for i in len(w):
    data.append(w[i])

In [ ]:
col = list(w.columns)

In [ ]:

data = [tuple(item) for item in list(w.to_numpy())]

In [ ]:
ll

In [ ]:
w["Name"] = w["Name"].astype("string")
w["place"] = w["place"].astype("string")

In [ ]:
w.dtypes

In [ ]:
w

In [ ]:
from pyspark.sql.types import StructType,StructField, StringType, IntegerType


schema = StructType([
    StructField("Name", StringType(), True), \
    StructField("place", StringType(), True)
  ])
 
df = spark.createDataFrame(data=data,schema=schema)

In [ ]:
df

In [ ]:
df.show()

In [ ]:
val = [2,3,6,-5,10,1,1]
ev = []
odd = []

for item in val:
    if item%2==1:
        odd.append(item)



In [ ]:
# Python 3 program to find longest 
# even sum subsequence. 
INT_MIN = -100000000

# Returns sum of maximum even 
# sum subsequence 
def maxEvenSum(arr, n): 
	
	# Find sum of positive numbers 
	pos_sum = 0
	for i in range(n):
        if (arr[i] > 0):
        pos_sum += arr[i] 

	if (pos_sum % 2 == 0): 
        return pos_sum 

	ans = INT_MIN
    for i in range(n):
        if (arr[i] % 2 != 0):
            ans = max(ans, pos_sum + arr[i]) 
            
			# if (arr[i] > 0): 
			# 	ans = max(ans, pos_sum - arr[i]) 
			# else: 
			# 	ans = max(ans, pos_sum + arr[i]) 
            
	return ans 

a = [2,3,6,-5,10,1,1] 
n = len(a) 
print(maxEvenSum(a, n)) 

# This code is contributed by sahilshelangia 


In [ ]:
# Python 3 program to find lexicographically 
# largest and smallest substrings of size k.
def getSmallestAndLargest(s, k):
	
	# Initialize min and max as 
	# first substring of size k
	currStr = s[:k]
	lexMin = currStr
	lexMax = currStr

	# Consider all remaining substrings. 
	# We consider every substring ending 
	# with index i.
	for i in range(k, len(s)):
		currStr = currStr[1 : k] + s[i]
		if (lexMax < currStr):
			lexMax = currStr
		if (lexMin >currStr):
			lexMin = currStr

	# Print result.
	print(lexMin)
	print(lexMax)

# Driver Code
if __name__ == '__main__':
	str1 = "0101101"
	k = 3
	getSmallestAndLargest(str1, k)

# This code is contributed by
# Surendra_Gangwar


In [ ]:
from pyspark.sql import SparkSession

# Create a Spark session
spark = SparkSession.builder \
    .appName("example") \
    .getOrCreate()

# Create a DataFrame with one column 'A' and one row with value 1
data = [(1,)]
df = spark.createDataFrame(data, ["A"])

# Show the DataFrame
df.show()

# Stop the Spark session
spark.stop()
